# R2-05 · Modelos basados en árboles — Lección

**Bootcamp de Datos para Funcionarios Públicos · Formación Pública**
Línea R2 · *Científico de Datos* · Semana 11 · Se ejecuta en Google Colab.

> 🚀 **Cómo se trabaja:** lee, ejecuta cada celda con **▶︎** (o `Shift`+`Enter`) y completa los `TODO`. Cada ejercicio termina en una **celda de chequeo** que muestra ✅ si está bien o una pista si todavía no.

---

## ¿Qué vas a lograr?

1. Entrenar e **interpretar** un árbol de decisión.
2. Ver y controlar el **sobreajuste** con la profundidad.
3. Entrenar un **bosque aleatorio** y leer la **importancia de variables**.
4. Comparar de forma justa contra el **baseline** y la regresión lineal de R2-04.
5. Convertir predicciones en **alertas** sobre la norma de 50 µg/m³.

**Competencia de salida:** entrenar, comparar e interpretar árboles y bosques aleatorios, controlando el sobreajuste.

**Dato real:** PM2.5 diario, temperatura y humedad de Cerro Navia 2020–2025 (SINCA, MMA) — el mismo problema de R2-04.

**Entregable:** que las **5 celdas de chequeo** muestren ✅.

In [ ]:
# ── Preparación del entorno (ejecuta esta celda primero) ──────────────────────
import os, urllib.request
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# "En vivo o caché": usa el archivo local; si falta (ej. Colab), lo baja del repo.
CSV = "aire_diario.csv"
if not os.path.exists(CSV):
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/{CSV}", CSV)

df = pd.read_csv(CSV, parse_dates=["fecha"])
cn = df[df["estacion"] == "Cerro Navia"].sort_values("fecha").reset_index(drop=True)

# Las mismas features de R2-04 (ya fabricadas para ti)
cn["pm25_ayer"] = cn["pm25"].shift(1)
cn["mes"] = cn["fecha"].dt.month
cn["dia_semana"] = cn["fecha"].dt.dayofweek
FEATURES = ["temp_media", "humedad_media", "mes", "dia_semana", "pm25_ayer"]
datos = cn.dropna(subset=FEATURES + ["pm25"])
print("Cerro Navia, días completos:", len(datos))

## Por qué el mismo dato (a propósito)

En R2-04 pronosticaste el PM2.5 de Cerro Navia con una **recta**. Hoy usarás **árboles** sobre
**exactamente el mismo problema**: mismas features, mismo corte temporal. Así la comparación es
justa y te concentras en lo nuevo — el modelo — sin distraerte con otro dataset.

| Vara de R2-04 (MAE en 2025) | µg/m³ |
| --- | --- |
| Baseline «mañana = hoy» | ≈ 7,36 |
| Regresión lineal | ≈ 7,09 |

Un **árbol de decisión** predice con una **secuencia de preguntas sí/no**: *¿ayer hubo más de
33,5 µg/m³? → ¿hoy hace menos de 9 °C? → …* hasta una hoja con un valor estimado.

> 🧠 **Analogía.** Es un flujograma de trámite: «si es invierno, pase a la ventanilla A; si además
> ayer hubo episodio, …». Por eso los árboles son de los modelos más fáciles de explicar a una autoridad.

## 1. Preparar: el mismo corte temporal, las mismas varas

Repite el corte de R2-04 (entrenar 2020–2024, probar 2025) y recalcula las dos varas: el baseline de persistencia y la regresión lineal.

### ✍️ Ejercicio 1 — Corte temporal y MAE de referencia

In [ ]:
corte = pd.Timestamp("2025-01-01")
train = datos[datos["fecha"] < corte]
test = datos[datos["fecha"] >= corte]
X_train, y_train = train[FEATURES], train["pm25"]
X_test, y_test = test[FEATURES], test["pm25"]
mae_base = mean_absolute_error(y_test, X_test["pm25_ayer"])
mae_lr = mean_absolute_error(y_test, LinearRegression().fit(X_train, y_train).predict(X_test))
print("entrenamiento:", len(train), "| prueba:", len(test))
print("MAE baseline:", round(mae_base, 2), "| MAE regresión lineal:", round(mae_lr, 2))

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert train["fecha"].max() < test["fecha"].min()
    assert len(train) + len(test) == len(datos)
    assert mae_lr < mae_base
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'train = datos[datos["fecha"] < corte] y test = datos[datos["fecha"] >= corte] (igual que en R2-04).')
    print("   Detalle:", e)

## 2. Tu primer árbol

El hiperparámetro **`max_depth`** limita cuántas preguntas seguidas puede hacer el árbol. Empieza con uno chico, `max_depth=3` (8 hojas como máximo), y fija `random_state=42` para que el resultado sea reproducible.

### ✍️ Ejercicio 2 — Entrena un árbol de profundidad 3 y mide su MAE en 2025

In [ ]:
arbol = DecisionTreeRegressor(max_depth=3, random_state=42)
arbol.fit(X_train, y_train)
mae_arbol = mean_absolute_error(y_test, arbol.predict(X_test))
print("MAE árbol (profundidad 3):", round(mae_arbol, 2), "| lineal:", round(mae_lr, 2), "| baseline:", round(mae_base, 2))

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    assert arbol.get_depth() == 3
    assert abs(mae_arbol - mean_absolute_error(y_test, arbol.predict(X_test))) < 1e-9
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'DecisionTreeRegressor(max_depth=3, random_state=42); .fit(X_train, y_train); mean_absolute_error(y_test, arbol.predict(X_test)).')
    print("   Detalle:", e)

In [ ]:
# (ilustración) Así «piensa» tu árbol: cada caja es una pregunta; las hojas, el PM2.5 estimado
try:
    plt.figure(figsize=(16, 6))
    plot_tree(arbol, feature_names=FEATURES, filled=True, rounded=True, fontsize=8, impurity=False)
    plt.title("Árbol de profundidad 3 — PM2.5 de Cerro Navia"); plt.show()
except (NameError, TypeError, ValueError, AttributeError):
    print("ℹ️ Completa el ejercicio 2 para ver este gráfico.")

## 3. El sobreajuste, ahora visible

Un árbol **sin límite** de profundidad sigue preguntando hasta dejar casi cada día de entrenamiento en su propia hoja: el error de entrenamiento cae a ~0… y el de prueba se dispara. Es la firma del **sobreajuste**: memorizar en vez de aprender.

### ✍️ Ejercicio 3 — Árbol sin límite: MAE en entrenamiento vs prueba

In [ ]:
arbol_profundo = DecisionTreeRegressor(random_state=42).fit(X_train, y_train)
mae_train = mean_absolute_error(y_train, arbol_profundo.predict(X_train))
mae_test = mean_absolute_error(y_test, arbol_profundo.predict(X_test))
print("profundidad alcanzada:", arbol_profundo.get_depth(), "| hojas:", arbol_profundo.get_n_leaves())
print("MAE entrenamiento:", round(mae_train, 2), "| MAE prueba:", round(mae_test, 2))

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    assert arbol_profundo.get_depth() > 10, "El árbol no debe tener límite de profundidad"
    assert mae_train < 0.5, "Sin límite, el árbol casi memoriza el entrenamiento"
    assert mae_test > mae_arbol, "…y en datos nuevos le va peor que al árbol chico"
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'DecisionTreeRegressor(random_state=42).fit(X_train, y_train); mide el MAE con predict(X_train) y con predict(X_test).')
    print("   Detalle:", e)

In [ ]:
# (ilustración) El error según la profundidad: entrenamiento baja siempre; prueba, no
try:
    prof = list(range(1, 16))
    err_tr, err_te = [], []
    for d in prof:
        m = DecisionTreeRegressor(max_depth=d, random_state=42).fit(X_train, y_train)
        err_tr.append(mean_absolute_error(y_train, m.predict(X_train)))
        err_te.append(mean_absolute_error(y_test, m.predict(X_test)))
    plt.figure(figsize=(7, 4))
    plt.plot(prof, err_tr, "o-", color="#4240e5", label="entrenamiento (memoria)")
    plt.plot(prof, err_te, "s-", color="#c0392b", label="prueba 2025 (generalización)")
    plt.axhline(mae_base, color="gray", ls=":", label="baseline «mañana = hoy»")
    plt.xlabel("max_depth"); plt.ylabel("MAE (µg/m³)"); plt.title("Más profundo no es mejor")
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
except (NameError, TypeError, ValueError, AttributeError):
    print("ℹ️ Completa el ejercicio 1 para ver este gráfico.")

## 4. Bosque aleatorio e importancia de variables

Un **bosque aleatorio** entrena muchos árboles, cada uno con una muestra distinta de días y de features, y **promedia** sus predicciones: la sabiduría de la multitud. Con `min_samples_leaf=5` cada hoja debe tener al menos 5 días, otra forma de frenar la memorización.

De regalo trae `feature_importances_`: **cuánto pesó cada variable**. A veces saber qué manda vale más que la predicción misma.

### ✍️ Ejercicio 4 — Entrena el bosque, mide su MAE y encuentra la feature más importante

In [ ]:
bosque = RandomForestRegressor(n_estimators=100, min_samples_leaf=5, random_state=42)
bosque.fit(X_train, y_train)
pred_bosque = bosque.predict(X_test)
mae_bosque = mean_absolute_error(y_test, pred_bosque)
importancias = pd.Series(bosque.feature_importances_, index=FEATURES).sort_values(ascending=False)
feature_top = importancias.idxmax()
print("MAE bosque:", round(mae_bosque, 2), "| lineal:", round(mae_lr, 2), "| baseline:", round(mae_base, 2))
print(importancias.round(3))

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    assert hasattr(bosque, "estimators_") and len(bosque.estimators_) == 100
    assert mae_bosque < mae_lr < mae_base, "El bosque debería superar a la recta y al baseline"
    assert feature_top == "pm25_ayer"
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'bosque.fit(X_train, y_train); la más importante es importancias.idxmax().')
    print("   Detalle:", e)

In [ ]:
# (ilustración) ¿Qué variable manda?
try:
    plt.figure(figsize=(6, 3.2))
    importancias.sort_values().plot.barh(color="#4240e5")
    plt.xlabel("importancia (suma 1)"); plt.title("Bosque aleatorio — importancia de variables")
    plt.tight_layout(); plt.show()
except (NameError, TypeError, ValueError, AttributeError):
    print("ℹ️ Completa el ejercicio 4 para ver este gráfico.")

## 5. De la predicción a la alerta

Una autoridad no necesita el µg/m³ exacto: necesita saber **qué días avisar**. Convierte las predicciones del bosque en alertas («predice más de 50») y cuéntalas contra los días que **realmente** superaron la norma en 2025. Compáralo con alertar usando el baseline.

### ✍️ Ejercicio 5 — Días detectados y falsas alarmas del bosque

In [ ]:
real = (y_test > 50).values
alerta = pred_bosque > 50
dias_reales = int(real.sum())
detectados = int((alerta & real).sum())
falsas_alarmas = int((alerta & ~real).sum())
alerta_base = (X_test["pm25_ayer"] > 50).values
print("días reales sobre 50:", dias_reales)
print("bosque   → detectados:", detectados, "| falsas alarmas:", falsas_alarmas)
print("baseline → detectados:", int((alerta_base & real).sum()), "| falsas alarmas:", int((alerta_base & ~real).sum()))

In [ ]:
# ── Celda de chequeo · Ejercicio 5 ──────────────────────────────────────────
try:
    assert dias_reales == int((y_test > 50).sum())
    assert detectados == int(((pred_bosque > 50) & (y_test.values > 50)).sum())
    assert falsas_alarmas == int(((pred_bosque > 50) & (y_test.values <= 50)).sum())
    assert 0 < detectados <= dias_reales
    print("✅ Ejercicio 5: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'alerta = pred_bosque > 50; detectados = int((alerta & real).sum()); falsas_alarmas = int((alerta & ~real).sum()).')
    print("   Detalle:", e)

## ✍️ Reflexión (no se corrige, pero es lo más importante)

En el ejercicio 5, el bosque da **menos falsas alarmas** que el baseline, pero se le escapan **más**
días reales sobre la norma. ¿Qué error es más caro para la salud pública: avisar de más o avisar de
menos? ¿Moverías el umbral de alerta por debajo de 50? (En R2-06 y R2-07 esto tiene nombre:
*precision* y *recall*.)

## Errores típicos

- **No fijar `max_depth`** (ni `min_samples_leaf`) y sorprenderse con el sobreajuste.
- **Elegir la profundidad mirando la prueba** → en rigor, se elige con un conjunto de **validación** aparte (R2-07).
- **Leer la importancia como causalidad** → `pm25_ayer` manda porque el aire persiste, no porque «cause» el de hoy.

## Cierre

- Un **árbol** predice con preguntas sí/no: interpretable, pero inestable y propenso a sobreajustar.
- La **profundidad** controla la complejidad: el error de entrenamiento siempre baja; el de prueba, no.
- El **bosque aleatorio** promedia muchos árboles: aquí supera a la recta y al baseline.
- La **importancia de variables** dice qué manda; la **alerta** convierte el modelo en decisión.

> *En R2-06 pasarás de predecir números a predecir categorías (clasificación) y a agrupar sin etiquetas (clustering).*